# Project E.C.H.O. - Radar Target Artificial Intelligence

This notebook builds the ultra-lightweight Neural Network designed to operate on an **ESP8266 Microcontroller**.
It loads massive raw FMCW radar streams from the Xenodo dataset and perfectly compresses the logic into a native C++ format.

In [4]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

### 1. Organize File Storage Paths
We point to the localized 77GHz dataset containing Birds, Drones, and Human footprints.

In [5]:
import os
import numpy as np
import tensorflow as tf # Ensure tensorflow is imported, although it's in the previous cell.

# Define Colab-friendly paths
EXPORT_DIR = "/content/ai_workflow/exported_models"

if not os.path.exists(EXPORT_DIR):
    os.makedirs(EXPORT_DIR)

# Zenodo dataset details
DATASET_DOWNLOAD_URL = "https://zenodo.org/records/5845259/files/data_SAAB_SIRS_77GHz_FMCW.npy?download=1"
DATASET_FILENAME = "data_SAAB_SIRS_77GHz_FMCW.npy"

print(f"Downloading dataset from: {DATASET_DOWNLOAD_URL}")
# tf.keras.utils.get_file downloads and caches the file, returning its path
DATASET_PATH = tf.keras.utils.get_file(DATASET_FILENAME, DATASET_DOWNLOAD_URL)

print(f"Loading massive dataset from: {DATASET_PATH}")
data = np.load(DATASET_PATH, allow_pickle=True)


1555196683/1555196683 ━━━━━━━━━━━━━━━━━━━━ 535s 0us/step
Loading massive dataset from: /root/.keras/datasets/data_SAAB_SIRS_77GHz_FMCW.npy


### 2. Multi-Class Separation & Micro-Doppler Extraction
Radar complex matrices are decoded into pure real amplitudes perfectly formatted for the TinyML CNN engine. Drones, Birds, and Humans are segregated.

In [6]:
drones = ['D1', 'D2', 'D3', 'D4', 'D5', 'D6']
birds = ['seagull', 'pigeon', 'raven', 'black-headed gull', 'heron', 'seagull and black-headed gull']
humans = ['human_walk', 'human_run']

X_list = []
y_list = []

print("Extracting classes from Complex radar numbers...")
for row in data:
    label_str = row[0]
    if label_str in birds:
        label = 0
    elif label_str in drones:
        label = 1
    elif label_str in humans:
        label = 2
    else:
        continue # Ignore corner reflectors and noise

    complex_data = row[1]
    real_magnitude_matrix = np.abs(complex_data)
    num_segments = real_magnitude_matrix.shape[1]
    for i in range(num_segments):
        single_segment = real_magnitude_matrix[:, i]
        X_list.append(single_segment)
        y_list.append(label)

X = np.array(X_list)
y = np.array(y_list)
X, y = shuffle(X, y, random_state=42)

Extracting classes from Complex radar numbers...


### 3. ESP8266 Memory Normalization
Instead of deploying complex StandardScaler Z-scores on the ESP8266 CPU, we simply use the global max absolute magnitude. This yields a single fractional division per sensor reading.

In [7]:
global_max_val = np.max(X)
X = X / global_max_val
X = X.reshape((X.shape[0], X.shape[1], 1))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Extracted {len(X_train)} training samples and {len(X_test)} testing samples.")
print(f"Data Shape: {X_train.shape}")
print(f"IMPORTANT ESP8266 Normalization Value -> {global_max_val:.4f}")

Extracted 58070 training samples and 14518 testing samples.
Data Shape: (58070, 1280, 1)
IMPORTANT ESP8266 Normalization Value -> 143.2784


### 4. Convolutional Deep Learning Array
This 2-Layer CNN relies on heavy kernel filters alongside robust Dropout algorithms to extract hidden micro-Doppler sub-patterns without overwhelming Microcontroller memory space.

In [8]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv1D(filters=32, kernel_size=16, strides=4, activation='relu', input_shape=(1280, 1)),
    tf.keras.layers.MaxPooling1D(pool_size=4),

    tf.keras.layers.Conv1D(filters=64, kernel_size=8, strides=4, activation='relu'),
    tf.keras.layers.GlobalAveragePooling1D(),

    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax')
])

adam_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=adam_optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 317, 32)        │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 79, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 18, 64)         │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,171 (74.89 KB)

 Trainable params: 19,171 (74.89 KB)

 Non-trainable params: 0 (0.00 B)

### 5. Automated Intelligence Modeling (Early Stopping)
Executing standard Adam backpropagation gradients with an Early Stopping safety net to freeze weights at their literal peak analytical capability.

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=200, batch_size=128, validation_data=(X_test, y_test), callbacks=[early_stop])

Epoch 1/200
454/454 ━━━━━━━━━━━━━━━━━━━━ 22s 42ms/step - accuracy: 0.8078 - loss: 0.7651 - val_accuracy: 0.8079 - val_loss: 0.6143
Epoch 2/200
454/454 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - accuracy: 0.8131 - loss: 0.5822 - val_accuracy: 0.8144 - val_loss: 0.5289
Epoch 3/200
454/454 ━━━━━━━━━━━━━━━━━━━━ 19s 41ms/step - accuracy: 0.8152 - loss: 0.5170 - val_accuracy: 0.8144 - val_loss: 0.4811
Epoch 4/200
454/454 ━━━━━━━━━━━━━━━━━━━━ 18s 40ms/step - accuracy: 0.8130 - loss: 0.4835 - val_accuracy: 0.8149 - val_loss: 0.4615
Epoch 5/200
454/454 ━━━━━━━━━━━━━━━━━━━━ 20s 43ms/step - accuracy: 0.8146 - loss: 0.4661 - val_accuracy: 0.8148 - val_loss: 0.4497
Epoch 6/200
454/454 ━━━━━━━━━━━━━━━━━━━━ 19s 41ms/step - accuracy: 0.8209 - loss: 0.4555 - val_accuracy: 0.8148 - val_loss: 0.4435
Epoch 7/200
454/454 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - accuracy: 0.8292 - loss: 0.4482 - val_accuracy: 0.8390 - val_loss: 0.4417
Epoch 8/200
454/454 ━━━━━━━━━━━━━━━━━━━━ 18s 39ms/step - accuracy: 0.8337 - loss: 0

### 6. Validation Performance Graph Plotting

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy', color='green', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='blue', linestyle='dashed')
plt.title('E.C.H.O Model Accuracy (Birds vs Drones vs Humans)')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', color='red', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange', linestyle='dashed')
plt.title('E.C.H.O Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()

graph_path = os.path.join(EXPORT_DIR, "training_history_jupyter.png")
plt.savefig(graph_path)
plt.show()

### 7. TFLite Native C++ Transpiler
Compresses the mathematical object directly into an 8-bit quantized byte array that an Arduino `echo_model.h` library reads via EloquentTinyML.

In [ ]:
print("\nCompressing model into TFLite format...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

def convert_to_c_array(tflite_model_byte_array):
    hex_array = [hex(b) for b in tflite_model_byte_array]
    c_str = ""
    for i, h in enumerate(hex_array):
        c_str += f"{h}, "
        if (i + 1) % 12 == 0:
            c_str += "\n  "
    return c_str

cpp_path = os.path.join(EXPORT_DIR, "echo_model_jupyter.h")
with open(cpp_path, "w") as f:
    f.write(f"// E.C.H.O. Automated Target Recognition Pipeline\n")
    f.write(f"// Global Normalization Divisor: {global_max_val:.4f}\n\n")
    f.write("#ifndef ECHO_MODEL_H\n")
    f.write("#define ECHO_MODEL_H\n\n")
    f.write("alignas(8) const unsigned char echo_model_tflite[] = {\n  ")
    f.write(convert_to_c_array(tflite_model))
    f.write("\n};\n")
    f.write(f"const unsigned int echo_model_tflite_len = {len(tflite_model)};\n\n")
    f.write("#endif // ECHO_MODEL_H\n")

print("SUCCESS! C++ Header created for ESP8266 deployment.")

### 8. Save Model in H5 Format
We will save the trained Keras model in the `.h5` format, which is a standard way to save Keras models, including their architecture, weights, and optimizer state. This will allow for easy loading and further use of the trained model.

In [ ]:
import os

# Ensure EXPORT_DIR is defined, as it might not be in this cell's scope if only this cell is run.
# In the full notebook, it's defined earlier.
if 'EXPORT_DIR' not in globals():
    EXPORT_DIR = "/content/ai_workflow/exported_models"
    if not os.path.exists(EXPORT_DIR):
        os.makedirs(EXPORT_DIR)

model_h5_path = os.path.join(EXPORT_DIR, "echo_model_jupyter.h5")
model.save(model_h5_path)
print(f"Keras model saved to: {model_h5_path}")